# HealthConnect Clinic - Week 4 Initial Analysis (Data Analytics Track)

**Programme:** AnalystLab Africa Experience Lab  
**Prepared by:** Lusani Judith Nempumbuluni  

Purpose: review the dataset structure, assess data quality, explore variables relevant to appointment attendance, and produce a reproducible cleaned dataset for Week 5.

The raw file is never modified. All outputs are written to `data/processed` and `figures`.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 50)

RAW = '../data/raw/HealthConnect_Appointment_Data.csv'
df = pd.read_csv(RAW)
df.shape

## 1. Dataset overview

In [ ]:
df.head()

In [ ]:
df.info()
df.describe(include='number').T

In [ ]:
print('Records:', len(df))
print('Unique patients:', df.patient_id.nunique())
for col in ['gender','age_group','appointment_type','appointment_day','appointment_time',
            'reminder_sent','reminder_channel','appointment_outcome']:
    print('\n', col)
    print(df[col].value_counts(dropna=False))

## 2. Data quality assessment
### 2.1 Completeness

In [ ]:
quality = pd.DataFrame({'dtype': df.dtypes.astype(str),
                        'missing': df.isna().sum(),
                        'missing_pct': (df.isna().mean()*100).round(2),
                        'unique': df.nunique()})
quality

### 2.2 Validity and consistency checks

In [ ]:
bd = pd.to_datetime(df.booking_date, format='%m/%d/%Y')
ad = pd.to_datetime(df.appointment_date, format='%m/%d/%Y')

def band(a):
    return '18-24' if a<25 else '25-34' if a<35 else '35-44' if a<45 else '45-54' if a<55 else '55-64' if a<65 else '65+'

checks = {
 'duplicate appointment_id': df.appointment_id.duplicated().sum(),
 'duplicate rows': df.duplicated().sum(),
 'negative lead time': ((ad-bd).dt.days < 0).sum(),
 'lead days mismatch': (((ad-bd).dt.days) != df.booking_lead_days).sum(),
 'weekday mismatch': (ad.dt.day_name() != df.appointment_day).sum(),
 'age_group mismatch': (df.age.map(band) != df.age_group).sum(),
 'no_shows > appointments': (df.previous_no_shows > df.previous_appointments).sum(),
 'channel set but no reminder': ((df.reminder_sent=='No') & df.reminder_channel.notna()).sum(),
 'reminder sent but no channel': ((df.reminder_sent=='Yes') & df.reminder_channel.isna()).sum(),
}
pd.Series(checks, name='records_affected').to_frame()

**Result:** every check returns zero. The dataset is internally consistent; only `distance_to_clinic_km` (1.8%), `waiting_time_minutes` (1.2%) and the structurally missing `reminder_channel` (27.3%) need treatment.

## 3. Cleaning and feature preparation

In [ ]:
c = df.copy()
c['booking_date'] = bd.dt.strftime('%Y-%m-%d')
c['appointment_date'] = ad.dt.strftime('%Y-%m-%d')
c['reminder_channel'] = c['reminder_channel'].fillna('None')
c['distance_to_clinic_km_missing'] = df.distance_to_clinic_km.isna().astype(int)
c['waiting_time_minutes_missing'] = df.waiting_time_minutes.isna().astype(int)
c['appointment_month'] = ad.dt.to_period('M').astype(str)
c['attended_flag'] = (c.appointment_outcome=='Attended').astype(int)
c['no_show_flag'] = (c.appointment_outcome=='No-Show').astype(int)
c['cancelled_flag'] = (c.appointment_outcome=='Cancelled').astype(int)
c['prior_no_show_rate'] = np.where(c.previous_appointments>0, (c.previous_no_shows/c.previous_appointments).round(3), np.nan)
c['lead_time_band'] = pd.cut(c.booking_lead_days, [-1,7,14,30,45,1000],
                             labels=['0-7 days','8-14 days','15-30 days','31-45 days','46+ days'])
c['distance_band'] = pd.cut(c.distance_to_clinic_km, [-1,5,10,20,1000],
                            labels=['0-5 km','5-10 km','10-20 km','20+ km'])
c.to_csv('../data/processed/HealthConnect_Appointment_Data_Clean.csv', index=False)
c.head()

## 4. Initial exploration (Week 4 signals only)

In [ ]:
outcome = c.appointment_outcome.value_counts(normalize=True).mul(100).round(1)
outcome

In [ ]:
c.groupby('lead_time_band', observed=True).no_show_flag.mean().mul(100).round(1)

In [ ]:
c.groupby('reminder_channel').no_show_flag.mean().mul(100).round(1)

In [ ]:
c.groupby('appointment_type').no_show_flag.mean().mul(100).round(1)

In [ ]:
c.groupby('age_group').no_show_flag.mean().mul(100).round(1)

In [ ]:
c.groupby('appointment_outcome')[['booking_lead_days','previous_no_shows',
                                  'distance_to_clinic_km','waiting_time_minutes','age']].mean().round(2)

## 5. Week 4 conclusions

* 48.5% no-show rate - the clinic loses almost half of its booked capacity.
* Data quality is strong; only two variables need a missing-value decision.
* Booking lead time is the strongest early signal; prior no-show history is the second.
* Reminders help only modestly, so reminders alone will not solve the problem.

**Week 5:** calculate the five proposed KPIs, run statistical testing on each driver, and build the Power BI dashboard.